In [1]:
import os
import copy
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from torchvision.models import resnet101
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
import torch.nn.functional as F
import time

In [2]:
# Set constants and paths
data_dir = "image-results-spoofing/data"
train_dir = os.path.join(data_dir, "Train")
test_dir = os.path.join(data_dir, "Test")
batch_size = 32
learning_rate = 0.001
num_epochs = 20
RANDOM_STATE = 1082
MODEL_NAME = "ResNet-LSTM"
DATASET_NAME = "spoofing-merged-gps-only.csv"
RESULTS_CSV = "../../UAVATTACK_Results.csv"

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Set numpy random seed
np.random.seed(RANDOM_STATE)
# Set torch random seed
torch.manual_seed(RANDOM_STATE)
# If using CUDA
if torch.cuda.is_available():
    torch.cuda.manual_seed(RANDOM_STATE)
    torch.cuda.manual_seed_all(RANDOM_STATE)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [4]:
train_transforms = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((82, 64)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])
test_transforms = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((82, 64)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])

In [5]:
# Datasets and DataLoaders
train_full = datasets.ImageFolder(root=train_dir, transform=train_transforms)
val_full = datasets.ImageFolder(root=train_dir, transform=test_transforms)
test_dataset = datasets.ImageFolder(root=test_dir, transform=test_transforms)

train_idx, val_idx = train_test_split(
    np.arange(len(train_full)), train_size=0.8, random_state=RANDOM_STATE,
    stratify=np.array(train_full.targets)
)
train_dataset = Subset(train_full, train_idx)
val_dataset = Subset(val_full, val_idx)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=0)

dataloaders = {'train': train_loader, 'val': val_loader, 'test': test_loader}
classes = train_full.classes

In [6]:
class CNN_LSTM_Model(nn.Module):
    def __init__(self, num_classes=2, lstm_hidden=128, lstm_layers=1):
        super().__init__()
        resnet = resnet101(pretrained=True)
        # Change first conv to accept grayscale
        resnet.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)

        self.features = nn.Sequential(
            resnet.conv1,
            resnet.bn1,
            resnet.relu,
            resnet.maxpool,
            resnet.layer1,
            resnet.layer2,
            resnet.layer3,
            resnet.layer4
        )
        # Find out the output shape by running a dummy input
        with torch.no_grad():
            dummy = torch.zeros(1, 1, 82, 64)
            dummy_feats = self.features(dummy)
        B, C, H, W = dummy_feats.shape
        self.lstm = nn.LSTM(
            input_size=W*C, hidden_size=lstm_hidden,
            num_layers=lstm_layers, batch_first=True
        )
        self.classifier = nn.Sequential(
            nn.Linear(lstm_hidden, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        feats = self.features(x)
        feats = feats.permute(0, 2, 1, 3)
        feats = feats.contiguous().view(x.size(0), feats.size(1), -1)
        lstm_out, (h_n, _) = self.lstm(feats)
        out = h_n[-1]
        out = self.classifier(out)
        return out

In [7]:
# Instantiate model, loss, optimizer
model = CNN_LSTM_Model(num_classes=2).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=learning_rate, momentum=0.9)

C:\Users\antho\miniforge3\envs\pytorch\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\antho\miniforge3\envs\pytorch\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet101_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet101_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [8]:
# Training loop
best_model_wts = copy.deepcopy(model.state_dict())
best_acc = 0.0
best_epoch = 0

start_time = time.time()

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    all_train_preds = []
    all_train_labels = []
    for inputs, labels in dataloaders['train']:
        inputs = inputs.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)
        _, preds = torch.max(outputs, 1)
        all_train_preds.append(preds.cpu().numpy())
        all_train_labels.append(labels.cpu().numpy())
    epoch_loss = running_loss / len(train_dataset)
    train_preds = np.concatenate(all_train_preds)
    train_labels = np.concatenate(all_train_labels)
    train_acc = accuracy_score(train_labels, train_preds)

    # Validation on held-out validation set
    model.eval()
    all_val_preds = []
    all_val_labels = []
    with torch.no_grad():
        for inputs, labels in dataloaders['val']:
            inputs = inputs.to(device)
            labels = labels.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            all_val_preds.append(preds.cpu().numpy())
            all_val_labels.append(labels.cpu().numpy())
    val_preds = np.concatenate(all_val_preds)
    val_labels = np.concatenate(all_val_labels)
    val_acc = accuracy_score(val_labels, val_preds)
    print(f"Epoch {epoch+1}/{num_epochs} - Train Loss: {epoch_loss:.4f} - Train Acc: {train_acc:.4f} - Val Acc: {val_acc:.4f}")

    if val_acc > best_acc:
        best_acc = val_acc
        best_epoch = epoch + 1
        best_model_wts = copy.deepcopy(model.state_dict())

train_time = time.time() - start_time

model.load_state_dict(best_model_wts)

Epoch 1/20 - Train Loss: 0.5151 - Train Acc: 0.8499 - Val Acc: 0.8631
Epoch 2/20 - Train Loss: 0.4015 - Train Acc: 0.8624 - Val Acc: 0.8631
Epoch 3/20 - Train Loss: 0.3972 - Train Acc: 0.8624 - Val Acc: 0.8631
Epoch 4/20 - Train Loss: 0.3974 - Train Acc: 0.8624 - Val Acc: 0.8631
Epoch 5/20 - Train Loss: 0.3848 - Train Acc: 0.8624 - Val Acc: 0.8631
Epoch 6/20 - Train Loss: 0.3732 - Train Acc: 0.8624 - Val Acc: 0.8631
Epoch 7/20 - Train Loss: 0.3348 - Train Acc: 0.8624 - Val Acc: 0.8631
Epoch 8/20 - Train Loss: 0.2832 - Train Acc: 0.8624 - Val Acc: 0.8631
Epoch 9/20 - Train Loss: 0.2322 - Train Acc: 0.8817 - Val Acc: 0.9129
Epoch 10/20 - Train Loss: 0.1772 - Train Acc: 0.9396 - Val Acc: 0.9557
Epoch 11/20 - Train Loss: 0.1288 - Train Acc: 0.9572 - Val Acc: 0.9571
Epoch 12/20 - Train Loss: 0.1173 - Train Acc: 0.9610 - Val Acc: 0.9516
Epoch 13/20 - Train Loss: 0.0758 - Train Acc: 0.9769 - Val Acc: 0.9737
Epoch 14/20 - Train Loss: 0.0732 - Train Acc: 0.9759 - Val Acc: 0.9640
Epoch 15/20 - T

<All keys matched successfully>

In [ ]:
# Final evaluation
model.eval()
all_preds = []
all_probs = []
all_labels = []
with torch.no_grad():
    for inputs, labels in dataloaders['test']:
        inputs = inputs.to(device)
        labels = labels.to(device)
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        all_probs.extend(torch.softmax(outputs, dim=1).cpu().numpy())
        all_preds.append(preds.cpu().numpy())
        all_labels.append(labels.cpu().numpy())
all_preds = np.concatenate(all_preds)
all_labels = np.concatenate(all_labels)

accuracy = accuracy_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds, average='weighted', zero_division=0)
print(f"Test Accuracy: {accuracy * 100:.2f}%")
print(f"Test F1 Score: {f1:.4f}")

In [ ]:
import os as _os
import sys as _sys

_root = _os.path.abspath(_os.getcwd())
while _root != _os.path.dirname(_root) and not _os.path.isfile(_os.path.join(_root, "eval_metrics.py")):
    _root = _os.path.dirname(_root)
if _root not in _sys.path:
    _sys.path.insert(0, _root)
from eval_metrics import append_row, evaluate, summary, time_inference_torch


import numpy as np

_timing = time_inference_torch(model, next(iter(dataloaders['val']))[0].to(device))

row = evaluate(
    all_labels,
    all_preds,
    model_name=MODEL_NAME,
    dataset_name=DATASET_NAME,
    family="Image",
    labels=list(range(len(classes))),
    class_names=classes,
    y_proba=all_probs,
    y_train=np.array(train_full.targets)[train_idx],
    train_time_sec=train_time,
    num_epochs=num_epochs,
    parameters=list(dict(model.named_parameters()).keys()),
    seed=RANDOM_STATE,
    n_features=int(np.prod(train_dataset[0][0].shape)),
    n_train=len(train_idx),
    timing=_timing,
)
append_row(row, RESULTS_CSV)
print(summary(row))
print(f"Results appended to {RESULTS_CSV}")

In [11]:
# Save best model
model_save_path = "best_cnn_lstm_model.pth"
checkpoint = {
    'epoch': best_epoch,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict()
}
torch.save(checkpoint, model_save_path)
print(f"Model saved to {model_save_path}")

Model saved to best_cnn_lstm_model.pth
